# 0006 - Zigzag Conversion

Neste notebook vamos resolver o problema **Zigzag Conversion** de forma bem didática,
indo da simulação mais literal até uma formulação matemática mais enxuta e eficiente.

A ideia é que este material funcione tanto como estudo guiado quanto como conteúdo pronto
para exportação em um **Jupyter Book**.


## Enunciado

### Enunciado original

Enunciado original

### Enunciado traduzido em linguagem simples

Enunciado traduzido em linguagem simples


## Imports

Colocar aqui todos os imports necessários


In [ ]:
from __future__ import annotations

from collections.abc import Callable
from time import perf_counter
import random
import string

import matplotlib.pyplot as plt
from IPython.display import display
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid", context="talk")
plt.rcParams["figure.figsize"] = (12, 5)
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False
plt.rcParams["axes.titleweight"] = "bold"


def validar_conversao(
    funcao: Callable[[str, int], str],
    casos: list[dict[str, object]],
) -> None:
    """Valida uma função de conversão Zigzag com casos conhecidos.

    Parâmetros:
    ----------
    funcao : Callable[[str, int], str]
        Função que será testada.
    casos : list[dict[str, object]]
        Lista com `s`, `numRows` e `esperado`.

    Retorno:
    -------
    None
        A função apenas executa asserções.
    """
    for caso in casos:
        resposta = funcao(str(caso["s"]), int(caso["numRows"]))
        assert resposta == caso["esperado"], (
            f"Falha para s={caso['s']!r}, numRows={caso['numRows']}: "
            f"esperado {caso['esperado']!r}, obtido {resposta!r}"
        )


def construir_curvas_complexidade(
    tamanhos: list[int],
    curvas: dict[str, Callable[[int], float]],
) -> pd.DataFrame:
    """Constrói um DataFrame com curvas teóricas de complexidade.

    Parâmetros:
    ----------
    tamanhos : list[int]
        Valores de n usados na comparação.
    curvas : dict[str, Callable[[int], float]]
        Dicionário com o nome da curva e uma função geradora.

    Retorno:
    -------
    pd.DataFrame
        Tabela com colunas `n`, `curva` e `valor`.
    """
    registros: list[dict[str, float | int | str]] = []
    for n in tamanhos:
        for nome_curva, funcao in curvas.items():
            registros.append({"n": n, "curva": nome_curva, "valor": float(funcao(n))})
    return pd.DataFrame(registros)


def plotar_curvas_complexidade(
    titulo: str,
    tamanhos: list[int],
    curvas: dict[str, Callable[[int], float]],
    legenda_extra: str = "",
) -> None:
    """Plota curvas teóricas de complexidade com seaborn.

    Parâmetros:
    ----------
    titulo : str
        Título principal do gráfico.
    tamanhos : list[int]
        Valores de n usados na comparação.
    curvas : dict[str, Callable[[int], float]]
        Curvas teóricas a serem exibidas.
    legenda_extra : str, default = ""
        Texto opcional abaixo do gráfico.
    """
    df_curvas = construir_curvas_complexidade(tamanhos, curvas)
    fig, axes = plt.subplots(1, 2, figsize=(15, 4.8), sharex=True)

    sns.lineplot(
        data=df_curvas,
        x="n",
        y="valor",
        hue="curva",
        marker="o",
        linewidth=2.2,
        ax=axes[0],
    )
    axes[0].set_title("Escala linear")
    axes[0].set_xlabel("Tamanho da entrada (n)")
    axes[0].set_ylabel("Custo relativo")

    sns.lineplot(
        data=df_curvas,
        x="n",
        y="valor",
        hue="curva",
        marker="o",
        linewidth=2.2,
        ax=axes[1],
        legend=False,
    )
    axes[1].set_yscale("log")
    axes[1].set_title("Escala logarítmica")
    axes[1].set_xlabel("Tamanho da entrada (n)")
    axes[1].set_ylabel("Custo relativo (log)")

    fig.suptitle(titulo, y=1.03, fontsize=15)
    if legenda_extra:
        fig.text(0.5, -0.02, legenda_extra, ha="center", fontsize=10)
    plt.tight_layout()
    plt.show()


def gerar_string_aleatoria(tamanho: int, seed: int = 42) -> str:
    """Gera uma string aleatória reprodutível para o benchmark.

    Parâmetros:
    ----------
    tamanho : int
        Comprimento da string gerada.
    seed : int, default = 42
        Semente para tornar a geração reprodutível.

    Retorno:
    -------
    str
        String gerada com letras e alguns sinais permitidos pelo problema.
    """
    rng = random.Random(seed)
    alfabeto = string.ascii_letters + ',.'
    return ''.join(rng.choice(alfabeto) for _ in range(tamanho))


def gerar_string_repetida(tamanho: int) -> str:
    """Gera uma string com caracteres repetidos.

    Parâmetros:
    ----------
    tamanho : int
        Comprimento da string gerada.

    Retorno:
    -------
    str
        String formada pela repetição de `"A"`.
    """
    return 'A' * tamanho


def gerar_string_padrao(tamanho: int, seed: int = 99) -> str:
    """Gera uma string com padrão misto para o benchmark.

    Parâmetros:
    ----------
    tamanho : int
        Comprimento da string gerada.
    seed : int, default = 99
        Semente para tornar a geração reprodutível.

    Retorno:
    -------
    str
        String com padrão textual variado.
    """
    rng = random.Random(seed + tamanho)
    base = 'PAYPALISHIRING,.'
    extra = ''.join(rng.choice(base) for _ in range(tamanho))
    return extra


def medir_funcao(
    funcao: Callable[[str, int], str],
    texto: str,
    num_rows: int,
    repeticoes: int = 5,
) -> tuple[float, float]:
    """Mede o tempo médio e o desvio padrão de uma função.

    Parâmetros:
    ----------
    funcao : Callable[[str, int], str]
        Função que será medida.
    texto : str
        String de entrada.
    num_rows : int
        Número de linhas usado na conversão.
    repeticoes : int, default = 5
        Quantidade de medições realizadas.

    Retorno:
    -------
    tuple[float, float]
        Tempo médio e desvio padrão, ambos em milissegundos.
    """
    if repeticoes < 1:
        raise ValueError('`repeticoes` deve ser pelo menos 1.')
    tempos: list[float] = []
    for _ in range(repeticoes):
        inicio = perf_counter()
        funcao(texto, num_rows)
        fim = perf_counter()
        tempos.append((fim - inicio) * 1000)
    serie = pd.Series(tempos, dtype='float64')
    return float(serie.mean()), float(serie.std(ddof=0))


def construir_casos_benchmark() -> list[dict[str, object]]:
    """Constrói os cenários usados no benchmark final.

    Retorno:
    -------
    list[dict[str, object]]
        Lista de cenários com tamanhos e quantidades de linhas variadas.
    """
    tamanhos = [10, 25, 50, 100, 250, 500]
    linhas = [1, 2, 3, 4, 5, 10]
    casos: list[dict[str, object]] = []

    for tamanho in tamanhos:
        for num_rows in linhas:
            casos.append(
                {
                    'cenario': 'Aleatória',
                    'entrada': f'n={tamanho}, r={num_rows}',
                    'texto': gerar_string_aleatoria(tamanho, seed=100 + tamanho + num_rows),
                    'tamanho': tamanho,
                    'numRows': num_rows,
                }
            )
            casos.append(
                {
                    'cenario': 'Repetida',
                    'entrada': f'n={tamanho}, r={num_rows}',
                    'texto': gerar_string_repetida(tamanho),
                    'tamanho': tamanho,
                    'numRows': num_rows,
                }
            )
            casos.append(
                {
                    'cenario': 'Padrão',
                    'entrada': f'n={tamanho}, r={num_rows}',
                    'texto': gerar_string_padrao(tamanho, seed=200 + tamanho + num_rows),
                    'tamanho': tamanho,
                    'numRows': num_rows,
                }
            )
    return casos


def executar_benchmark(
    funcoes: dict[str, Callable[[str, int], str]],
    casos: list[dict[str, object]],
    repeticoes: int = 5,
) -> pd.DataFrame:
    """Executa o benchmark sobre uma lista de casos.

    Parâmetros:
    ----------
    funcoes : dict[str, Callable[[str, int], str]]
        Mapeamento entre nome da solução e função.
    casos : list[dict[str, object]]
        Casos a serem avaliados.
    repeticoes : int, default = 5
        Quantidade de medições por caso.

    Retorno:
    -------
    pd.DataFrame
        Tabela com os tempos médios e desvios padrão.
    """
    registros: list[dict[str, object]] = []
    for nome_solucao, funcao in funcoes.items():
        for caso in casos:
            media_ms, desvio_ms = medir_funcao(
                funcao,
                str(caso['texto']),
                int(caso['numRows']),
                repeticoes=repeticoes,
            )
            registros.append(
                {
                    'solucao': nome_solucao,
                    'cenario': caso['cenario'],
                    'entrada': caso['entrada'],
                    'tamanho': int(caso['tamanho']),
                    'numRows': int(caso['numRows']),
                    'tempo_medio_ms': media_ms,
                    'desvio_padrao_ms': desvio_ms,
                }
            )
    return pd.DataFrame(registros)


casos_teste = [
    {'s': 'PAYPALISHIRING', 'numRows': 3, 'esperado': 'PAHNAPLSIIGYIR'},
    {'s': 'PAYPALISHIRING', 'numRows': 4, 'esperado': 'PINALSIGYAHRPI'},
    {'s': 'A', 'numRows': 1, 'esperado': 'A'},
    {'s': 'AB', 'numRows': 1, 'esperado': 'AB'},
    {'s': 'ABCD', 'numRows': 2, 'esperado': 'ACBD'},
    {'s': 'ABCDE', 'numRows': 4, 'esperado': 'ABCED'},
    {'s': 'ABCDEF', 'numRows': 3, 'esperado': 'AEBDFC'},
    {'s': 'ABC', 'numRows': 5, 'esperado': 'ABC'},
    {'s': 'A,B.C', 'numRows': 3, 'esperado': 'AC,.B'},
]


## Funções uteis

**Imports**

Colocar aqui todos os imports necessários

**Funções uteis**

**Intuição inicial**

O zigue-zague pode parecer um desenho em duas dimensões, mas a resposta final é uma string
linear. Isso é importante porque o processo visual ajuda a entender a lógica, porém o que
precisamos devolver no fim é apenas a concatenação das linhas.

A movimentação segue um padrão repetitivo:

- descemos linha por linha até chegar na última linha;
- depois subimos na diagonal até voltar à primeira linha;
- repetimos esse ciclo até consumir toda a string.

Exemplo com `numRows = 4`:

```text
desce:   0 -> 1 -> 2 -> 3
sobe:                2 -> 1 -> 0
```

Esse comportamento explica por que a solução tende a ser muito natural com uma estrutura de
linhas.

**Observações importantes**

Há dois casos especiais que precisam ser tratados logo no início:

- `numRows == 1`: não existe zigue-zague de verdade, então a resposta é a própria string;
- `numRows >= len(s)`: cada caractere acaba caindo em uma linha diferente, então também
  retornamos `s` diretamente.

Esses atalhos deixam o código mais claro e evitam trabalho desnecessário.


## Solução 1 - Força bruta


### Descrição da solução

**Descrição da solução**

Descrição intuitiva da solução.

**Implementação da força bruta**


### Ordem de complexidade (memória e processamento)

**Ordem de complexidade (memória e processamento)**

- **Tempo:** `O(n * numRows)` no pior caso, porque a matriz inteira precisa ser criada,
  preenchida e lida.
- **Memória:** `O(n * numRows)`, já que a grade 2D guarda muitos espaços vazios.

Essa é a solução mais visual, mas também a menos econômica. Ela serve muito bem para entender
o problema, porém não é a melhor escolha quando pensamos em eficiência.


### Mini walkthrough

Exibição de alguns exemplos de como o algoritmo funciona.


### Exemplo de execuçaõ em ASCII sketch

**Exemplo de execuçaõ em ASCII sketch**

```text
s = PAYPALISHIRING, numRows = 4

Movimento:
desce:   0 -> 1 -> 2 -> 3
sobe:                2 -> 1 -> 0

Grade conceitual:
P       I       N
A     L S     I G
Y   A   H   R
P       I
```

A grade completa deixa o desenho muito intuitivo, mas também revela o custo de manter uma
estrutura 2D maior do que o necessário.


### Implementação da força bruta


In [ ]:
def convert_bruteforce_matrix(s: str, numRows: int) -> str:
    """Converte a string usando uma grade 2D explícita.

    Parâmetros:
    ----------
    s : str
        String de entrada.
    numRows : int
        Quantidade de linhas do zigzag.

    Retorno:
    -------
    str
        Resultado lido linha por linha.

    Exceções:
    --------
    Levanta TypeError se `s` não for uma string.
    """
    if not isinstance(s, str):
        raise TypeError('`s` deve ser uma string.')
    if numRows == 1 or numRows >= len(s):
        return s

    matriz = [[' '] * len(s) for _ in range(numRows)]
    linha = 0
    coluna = 0
    descendo = True

    for caractere in s:
        matriz[linha][coluna] = caractere
        if descendo:
            if linha == numRows - 1:
                descendo = False
                linha -= 1
                coluna += 1
            else:
                linha += 1
        else:
            if linha == 0:
                descendo = True
                linha += 1
            else:
                linha -= 1
                coluna += 1

    return ''.join(caractere for linha_atual in matriz for caractere in linha_atual if caractere != ' ')


### Testes


In [ ]:
validar_conversao(convert_bruteforce_matrix, casos_teste)
print('Todos os testes da solução 1 passaram.')


## Solução 2 - Melhorias da solução 1


### Descrição da solução

**Descrição da solução**

Descrição intuitiva da solução.

**Implementação da melhoria**


### Ordem de complexidade (memória e processamento)

**Ordem de complexidade (memória e processamento)**

- **Tempo:** `O(n)`, porque cada caractere é processado uma única vez.
- **Memória:** `O(n)`, já que as linhas acumulam todos os caracteres da saída.

Em termos práticos, essa solução já é muito boa: ela preserva a intuição visual e elimina o custo
extra da grade 2D.


### Mini walkthrough

Exibição de alguns exemplos de como o algoritmo funciona.


### Exemplo de execuçaõ em ASCII sketch

**Exemplo de execuçaõ em ASCII sketch**

```text
s = PAYPALISHIRING, numRows = 4

Linha 0: P     I     N
Linha 1: A   L S   I G
Linha 2: Y A   H R
Linha 3: P   I

Movimento:
0 -> 1 -> 2 -> 3 -> 2 -> 1 -> 0 -> 1 -> ...
```

A leitura visual fica quase tão clara quanto na força bruta, mas com muito menos memória.


### Implementação da melhoria


In [ ]:
def convert_row_buffers(s: str, numRows: int) -> str:
    """Converte a string usando uma lista de linhas.

    Parâmetros:
    ----------
    s : str
        String de entrada.
    numRows : int
        Quantidade de linhas do zigzag.

    Retorno:
    -------
    str
        Resultado final da conversão.
    """
    if not isinstance(s, str):
        raise TypeError('`s` deve ser uma string.')
    if numRows == 1 or numRows >= len(s):
        return s

    linhas = [''] * numRows
    linha = 0
    direcao = 1

    for caractere in s:
        linhas[linha] += caractere
        if linha == 0:
            direcao = 1
        elif linha == numRows - 1:
            direcao = -1
        linha += direcao

    return ''.join(linhas)


### Testes


In [ ]:
validar_conversao(convert_row_buffers, casos_teste)
print('Todos os testes da solução 2 passaram.')


## Solução 3 - Melhor solução


### Descrição da solução

**Descrição da solução**

Descrição intuitiva da solução.

**Implementação da melhoria**


### Ordem de complexidade (memória e processamento)

**Ordem de complexidade (memória e processamento)**

- **Tempo:** `O(n)`, porque cada caractere é visitado uma vez ao ser colocado na resposta.
- **Memória:** `O(n)`, por causa da string de saída.

Essa é a solução ideal porque combina clareza, eficiência e uma explicação matemática fácil de
acompanhar quando o padrão do ciclo já está entendido.


### Mini walkthrough

Exibição de alguns exemplos de como o algoritmo funciona.


### Exemplo de execuçaõ em ASCII sketch

**Exemplo de execuçaõ em ASCII sketch**

```text
s = PAYPALISHIRING, numRows = 4
ciclo = 6

Índices por linha:
linha 0 -> 0, 6, 12
linha 1 -> 1, 5, 7, 11, 13
linha 2 -> 2, 4, 8, 10
linha 3 -> 3, 9

Resultado: P I N / A L S I G / Y A H R / P I
```

Essa abordagem é elegante porque não depende de simular o vai-e-vem caractere por caractere;
ela organiza o problema com base na estrutura matemática do padrão.


### Implementação da melhoria


In [ ]:
def convert_cycle_math(s: str, numRows: int) -> str:
    """Converte a string usando o ciclo matemático do zigzag.

    Parâmetros:
    ----------
    s : str
        String de entrada.
    numRows : int
        Quantidade de linhas do zigzag.

    Retorno:
    -------
    str
        Resultado final da conversão.
    """
    if not isinstance(s, str):
        raise TypeError('`s` deve ser uma string.')
    if numRows == 1 or numRows >= len(s):
        return s

    ciclo = 2 * (numRows - 1)
    partes: list[str] = []

    for linha in range(numRows):
        for indice in range(linha, len(s), ciclo):
            partes.append(s[indice])
            diagonal = indice + ciclo - 2 * linha
            if 0 < linha < numRows - 1 and diagonal < len(s):
                partes.append(s[diagonal])

    return ''.join(partes)


### Testes


In [ ]:
validar_conversao(convert_cycle_math, casos_teste)
print('Todos os testes da solução 3 passaram.')


## Solução 4 - Solução enxuta para submissão no leetcode


### Descrição da solução

**Descrição da solução**

**Observação**

A lógica é a mesma da solução 3: calcular o ciclo do zigzag e coletar os caracteres de cada
linha na ordem correta.

**Implementação da melhoria**


### Ordem de complexidade (memória e processamento)

**Ordem de complexidade (memória e processamento)**

**Complexidade ilustrativa**

Os gráficos abaixo são didáticos. Eles não medem tempo real de execução; servem para mostrar
como o custo cresce de forma relativa nas abordagens.

Vamos usar uma comparação com `numRows = 4` como cenário representativo.


### Mini walkthrough

**Comparação entre as abordagens**

| Solução | Ideia central | Tempo | Memória | Prós | Contras |
|---|---|---:|---:|---|---|
| Força bruta | Construir uma grade 2D explícita | `O(n * numRows)` | `O(n * numRows)` | Muito visual e didática | Desperdiça memória |
| Melhorada | Manter apenas uma linha por faixa | `O(n)` | `O(n)` | Simples e intuitiva | Ainda depende de simulação |
| Melhor solução possível | Explorar o ciclo matemático do padrão | `O(n)` | `O(n)` | Elegante e eficiente | Exige entender o ciclo |
| Versão enxuta | Mesma lógica da solução ótima | `O(n)` | `O(n)` | Pronta para LeetCode | Menos explicativa |

A tabela mostra que a grande diferença está no custo extra da representação, não na resposta final.


### Exemplo de execuçaõ em ASCII sketch

**Exemplo de execuçaõ em ASCII sketch**

Exibição de algum exemplo do passo a passo do algoritmo.


### Implementação da melhoria


In [ ]:
class Solution:
    def convert(self, s: str, numRows: int) -> str:
        if numRows == 1 or numRows >= len(s):
            return s

        ciclo = 2 * (numRows - 1)
        partes: list[str] = []

        for linha in range(numRows):
            for indice in range(linha, len(s), ciclo):
                partes.append(s[indice])
                diagonal = indice + ciclo - 2 * linha
                if 0 < linha < numRows - 1 and diagonal < len(s):
                    partes.append(s[diagonal])

        return ''.join(partes)


In [ ]:
plotar_curvas_complexidade(
    titulo='Zigzag Conversion - crescimento teórico com numRows fixo em 4',
    tamanhos=list(range(1, 41)),
    curvas={
        'Solução 1 - Força bruta': lambda n: n * 4,
        'Solução 2 - Melhorada': lambda n: n,
        'Solução 3 - Ótima': lambda n: n,
    },
    legenda_extra='A solução 1 paga o custo da grade 2D; as soluções 2 e 3 crescem linearmente com o tamanho da string.',
)

linhas = [1, 2, 3, 4, 5, 10]
df_linhas = pd.DataFrame(
    {
        'numRows': linhas,
        'Solução 1 - Força bruta': [120 * r for r in linhas],
        'Solução 2 - Melhorada': [120 for _ in linhas],
        'Solução 3 - Ótima': [120 for _ in linhas],
    }
)

df_linhas_longo = df_linhas.melt(id_vars='numRows', var_name='curva', value_name='valor')
fig, ax = plt.subplots(figsize=(11, 5))
sns.lineplot(data=df_linhas_longo, x='numRows', y='valor', hue='curva', marker='o', ax=ax)
ax.set_title('Custo relativo ilustrativo em função de numRows (tamanho fixo)')
ax.set_xlabel('Número de linhas (numRows)')
ax.set_ylabel('Custo relativo')
plt.tight_layout()
plt.show()


### Testes


In [ ]:
solucao_leetcode = Solution()
validar_conversao(solucao_leetcode.convert, casos_teste)
print('Todos os testes da solução 4 passaram.')


## Benchmark

Construção de casos de testes para progressivos para mostrar
que as soluções se tornam cada vez mais otimizadas.
Exibir gráficos e tabelas de comparação entre as soluções pertinentes.

Construção de casos de testes para progressivos para mostrar
que as soluções se tornam cada vez mais otimizadas.
Exibir gráficos e tabelas de comparação entre as soluções pertinentes.

Agora vamos medir o desempenho real das três soluções didáticas em diferentes cenários.

A comparação usa strings curtas, médias e maiores, além de vários valores de `numRows`.
Assim, conseguimos observar tanto o impacto do tamanho da entrada quanto o impacto do número
 de linhas.

**Metodologia**

- cada caso é executado várias vezes para reduzir ruído;
- as entradas são reprodutíveis;
- a versão enxuta não entra no benchmark porque repete a solução ótima;
- o objetivo aqui é empírico, não formal.

**Interpretação dos resultados**

Os resultados tendem a seguir o comportamento esperado:

- a solução 1 cresce mais por causa da grade 2D;
- a solução 2 elimina o desperdício da matriz completa;
- a solução 3 costuma ser a mais estável e elegante, porque trabalha diretamente com o ciclo.

Como se trata de um benchmark empírico, pequenas inversões podem acontecer por ruído de
execução, mas a tendência geral deve permanecer clara.

**Conclusão**

O problema **Zigzag Conversion** é um ótimo exemplo de como uma ideia visual pode ser resolvida
de maneiras diferentes.

O caminho didático foi este:

1. começamos com uma simulação literal em grade 2D;
2. depois reduzimos o desperdício de memória usando apenas as linhas necessárias;
3. por fim, aproveitamos a estrutura cíclica do padrão para obter uma solução ótima e compacta.

Na prática:

- **para estudo:** as três primeiras soluções ajudam a construir intuição;
- **para entrevista:** a solução 3 é a mais interessante de explicar;
- **para submissão no LeetCode:** a solução 4 é a versão curta ideal.


In [ ]:
funcoes_benchmark = {
    '1. Força bruta': convert_bruteforce_matrix,
    '2. Melhorada': convert_row_buffers,
    '3. Ótima': convert_cycle_math,
}

casos_benchmark = construir_casos_benchmark()
df_benchmark = executar_benchmark(funcoes_benchmark, casos_benchmark, repeticoes=5)
df_benchmark = df_benchmark.sort_values(['cenario', 'numRows', 'tamanho', 'solucao']).reset_index(drop=True)

display(df_benchmark.head(15))

df_resumo = (
    df_benchmark
    .groupby(['cenario', 'solucao'], as_index=False)
    .agg(
        tempo_medio_ms=('tempo_medio_ms', 'mean'),
        desvio_padrao_ms=('desvio_padrao_ms', 'mean'),
        tamanho_medio=('tamanho', 'mean'),
        numRows_medio=('numRows', 'mean'),
    )
    .sort_values(['cenario', 'tempo_medio_ms'])
)

display(df_resumo)


In [ ]:
df_sintetico = df_benchmark.copy()

g = sns.relplot(
    data=df_sintetico,
    x='tamanho',
    y='tempo_medio_ms',
    hue='solucao',
    col='numRows',
    kind='line',
    marker='o',
    facet_kws={'sharey': False},
    col_wrap=3,
    height=4.0,
    aspect=1.15,
    linewidth=2.2,
)
g.set_axis_labels('Tamanho da entrada', 'Tempo médio (ms)')
g.set_titles('numRows = {col_name}')
g.fig.suptitle('Benchmark sintético por número de linhas', y=1.02, fontsize=16)
plt.show()

plt.figure(figsize=(13, 5))
sns.barplot(
    data=df_resumo,
    x='cenario',
    y='tempo_medio_ms',
    hue='solucao',
    errorbar=None,
)
plt.title('Tempo médio agregado por cenário')
plt.xlabel('Cenário')
plt.ylabel('Tempo médio (ms)')
plt.legend(title='Solução', bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.show()
